In [ ]:

import subprocess
import sys

# Check GPU availability
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(" GPU DETECTED:")
    print(result.stdout[:500])
except FileNotFoundError:
    print(" No GPU detected!")
    print("ACTION REQUIRED: Runtime → Change runtime type → T4 GPU → Save")
    print("Then: Runtime → Restart runtime")
    sys.exit("Please enable GPU first")

# Check Python version
import platform
print(f"\n Python version: {platform.python_version()}")
print(" Ready to proceed")

 GPU DETECTED:
Thu May  7 14:28:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|       

 Python version: 3.12.13
 Ready to proceed


In [ ]:
# CELL 2: Mount Google Drive

from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/drive/MyDrive'):
    print(" Google Drive mounted successfully")
    print(f"   Drive contents: {os.listdir('/content/drive/MyDrive')[:10]}")
else:
    print(" Drive mount failed. Try running this cell again.")

Mounted at /content/drive
 Google Drive mounted successfully
   Drive contents: ['Colab Notebooks', 'EEE241', 'maestro-v3.0.0-midi.zip', 'Task1_Outputs', 'maestro-v3.0.0', 'CSE425_Task3']


In [ ]:
# CELL 3A: Finding MAESTRO Zip in Drive and Extract It

import os
import zipfile
import glob
import shutil
from tqdm import tqdm

# ── Step 1: Search Drive for any MAESTRO-related zip file ──
DRIVE_ROOT = '/content/drive/MyDrive'
LOCAL_EXTRACT_DIR = '/content/maestro_extracted'  # Fast local Colab storage

print(" Searching Google Drive for MAESTRO zip file...")
print(f"   Scanning: {DRIVE_ROOT}")
print("   (This may take 10-20 seconds if your Drive has many files)\n")

all_zips = glob.glob(os.path.join(DRIVE_ROOT, '**', '*.zip'), recursive=True)

if not all_zips:
    print(" No .zip files found in your Google Drive.")
    print("\n   Possible fixes:")
    print("   1. Make sure the zip is fully uploaded to Drive (not still uploading)")
    print("   2. Check if the file is .tar.gz instead — tell your assistant")
    print("   3. Your Drive contents:")
    for item in os.listdir(DRIVE_ROOT)[:20]:
        print(f"      ├── {item}")
else:
    print(f"Found {len(all_zips)} zip file(s) in Drive:")
    for i, z in enumerate(all_zips):
        size_mb = os.path.getsize(z) / 1024 / 1024
        print(f"   [{i}] {z}  ({size_mb:.1f} MB)")

# ── Step 2: Identify the correct MAESTRO zip ──────────────
MAESTRO_ZIP = None

# Auto-detect: look for zip with 'maestro' in the name
for z in all_zips:
    if 'maestro' in os.path.basename(z).lower():
        MAESTRO_ZIP = z
        print(f"\n Auto-detected MAESTRO zip: {MAESTRO_ZIP}")
        break

# If auto-detection failed, pick the largest zip (likely MAESTRO)
if MAESTRO_ZIP is None and all_zips:
    all_zips_by_size = sorted(all_zips, key=os.path.getsize, reverse=True)
    MAESTRO_ZIP = all_zips_by_size[0]
    print(f"\n Could not auto-detect by name.")
    print(f"   Using largest zip file: {MAESTRO_ZIP}")
    print(f"   If this is WRONG, manually set:")
    print(f"   MAESTRO_ZIP = '/content/drive/MyDrive/EXACT_FILENAME.zip'")

if MAESTRO_ZIP is None:
    raise FileNotFoundError(
        "No zip file found. Upload your MAESTRO zip to Google Drive and re-run."
    )

# ── Step 3: Check if already extracted (skip if done) ─────
def find_maestro_root(search_dir):
    """Recursively find the folder containing maestro-v3.0.0.csv"""
    for root, dirs, files in os.walk(search_dir):
        if 'maestro-v3.0.0.csv' in files:
            return root
    return None

already_extracted = find_maestro_root(LOCAL_EXTRACT_DIR)

if already_extracted:
    MAESTRO_ROOT = already_extracted
    midi_count = len(glob.glob(os.path.join(MAESTRO_ROOT, '**', '*.midi'), recursive=True))
    midi_count += len(glob.glob(os.path.join(MAESTRO_ROOT, '**', '*.mid'), recursive=True))
    print(f"\n Already extracted at: {MAESTRO_ROOT}")
    print(f"   MIDI files found: {midi_count}")
else:
    # ── Step 4: Extract the zip ───────────────────────────
    zip_size_mb = os.path.getsize(MAESTRO_ZIP) / 1024 / 1024
    print(f"\n Extracting zip ({zip_size_mb:.1f} MB) to {LOCAL_EXTRACT_DIR} ...")
    print("  Expected time: 1–4 minutes depending on zip size")
    print("   Do NOT interrupt this cell.\n")

    os.makedirs(LOCAL_EXTRACT_DIR, exist_ok=True)

    with zipfile.ZipFile(MAESTRO_ZIP, 'r') as zf:
        members = zf.namelist()
        print(f"   Files in zip: {len(members)}")

        for member in tqdm(members, desc="Extracting", unit="file"):
            zf.extract(member, LOCAL_EXTRACT_DIR)

    print(f"\n Extraction complete.")

    # ── Step 5: Find MAESTRO_ROOT inside extracted folder ─
    MAESTRO_ROOT = find_maestro_root(LOCAL_EXTRACT_DIR)

    if MAESTRO_ROOT is None:
        # Maybe CSV has a different name — look for the folder with .midi files
        midi_dirs = set()
        for f in glob.glob(os.path.join(LOCAL_EXTRACT_DIR, '**', '*.midi'), recursive=True):
            midi_dirs.add(os.path.dirname(os.path.dirname(f)))
        if midi_dirs:
            MAESTRO_ROOT = min(midi_dirs, key=len)  # Shortest path = root
        else:
            MAESTRO_ROOT = LOCAL_EXTRACT_DIR
        print(f"  maestro-v3.0.0.csv not found at expected location.")
        print(f"   Using detected root: {MAESTRO_ROOT}")
    else:
        print(f" MAESTRO root found: {MAESTRO_ROOT}")

# ── Step 6: Final Verification ────────────────────────────
CSV_PATH = os.path.join(MAESTRO_ROOT, 'maestro-v3.0.0.csv')
midi_files_all = (
    glob.glob(os.path.join(MAESTRO_ROOT, '**', '*.midi'), recursive=True) +
    glob.glob(os.path.join(MAESTRO_ROOT, '**', '*.mid'),  recursive=True)
)

print(f"\n Verification:")
print(f"   MAESTRO_ROOT : {MAESTRO_ROOT}")
print(f"   CSV found    : {os.path.exists(CSV_PATH)}")
print(f"   MIDI files   : {len(midi_files_all)}")

if len(midi_files_all) >= 1000:
    print(f"\n SUCCESS — Dataset ready ({len(midi_files_all)} MIDI files)")
elif len(midi_files_all) > 0:
    print(f"\n Only {len(midi_files_all)} MIDI files found. Expected ~1276.")
    print(f"   Your zip may contain only a subset — this is fine, training will still work.")
else:
    print("\n No MIDI files found after extraction!")
    print(f"   Check extracted contents:")
    for item in os.listdir(LOCAL_EXTRACT_DIR)[:15]:
        print(f"   ├── {item}")
    print("\n  The zip may have a nested folder. Check the path manually.")

 Searching Google Drive for MAESTRO zip file...
   Scanning: /content/drive/MyDrive
   (This may take 10-20 seconds if your Drive has many files)

Found 1 zip file(s) in Drive:
   [0] /content/drive/MyDrive/maestro-v3.0.0-midi.zip  (55.7 MB)

 Auto-detected MAESTRO zip: /content/drive/MyDrive/maestro-v3.0.0-midi.zip

 Extracting zip (55.7 MB) to /content/maestro_extracted ...
  Expected time: 1–4 minutes depending on zip size
   Do NOT interrupt this cell.

   Files in zip: 1280


Extracting: 100%|██████████| 1280/1280 [00:02<00:00, 441.45file/s]


 Extraction complete.
 MAESTRO root found: /content/maestro_extracted/maestro-v3.0.0

 Verification:
   MAESTRO_ROOT : /content/maestro_extracted/maestro-v3.0.0
   CSV found    : True
   MIDI files   : 1276

 SUCCESS — Dataset ready (1276 MIDI files)


In [ ]:
# CELL 4: Install Libraries
import subprocess
import sys

packages = [
    'pretty_midi',
    'miditok',
    'tqdm',
]

print("Installing required packages...")
for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = "Ok" if result.returncode == 0 else "❌"
    print(f"  {status} {pkg}")

print("\nAll packages installed.")

Installing required packages...
  Ok pretty_midi
  Ok miditok
  Ok tqdm

All packages installed.


In [ ]:
# CELL 5: Importing All Libraries
import os
import glob
import json
import time
import pickle
import random
import warnings
warnings.filterwarnings('ignore')

# ── Numerical & Data ──────────────────────────────────────
import numpy as np
import pandas as pd

# ── MIDI Processing ───────────────────────────────────────
import pretty_midi

# ── PyTorch ───────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# ── Visualization ─────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for Colab
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# ── Progress & Utilities ──────────────────────────────────
from tqdm import tqdm
from collections import defaultdict, Counter

# ── Reproducibility ───────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device Setup ──────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" All libraries imported successfully")
print(f" Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f" PyTorch version: {torch.__version__}")
print(f" NumPy version: {np.__version__}")

 All libraries imported successfully
 Using device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB
 PyTorch version: 2.10.0+cu128
 NumPy version: 2.0.2


In [ ]:
# CELL 6: Global Configuration

print(f" MAESTRO_ROOT = {MAESTRO_ROOT}")
print(f" CSV_PATH     = {CSV_PATH}")

if 'MAESTRO_ROOT' not in dir() or not os.path.exists(MAESTRO_ROOT):
    raise RuntimeError(
        "MAESTRO_ROOT is not set. Please re-run Cell 3A first."
    )

# All other output folders go to Drive
OUTPUT_ROOT    = '/content/drive/MyDrive/Task1_Outputs'
CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, 'checkpoints')
PROCESSED_DIR  = os.path.join(OUTPUT_ROOT, 'processed_data')
MIDI_OUT_DIR   = os.path.join(OUTPUT_ROOT, 'generated_midi')
PLOTS_DIR      = os.path.join(OUTPUT_ROOT, 'plots')

for folder in [OUTPUT_ROOT, CHECKPOINT_DIR, PROCESSED_DIR, MIDI_OUT_DIR, PLOTS_DIR]:
    os.makedirs(folder, exist_ok=True)
print(" Output folders ready")

# ── Piano-Roll Parameters ─────────────────────────────────
FS            = 16       # Frames per second (time resolution)
WINDOW_LEN    = 128      # Time steps per window (128/16 = 8 seconds)
PIANO_LOW     = 21       # MIDI note 21 = A0 (lowest piano key)
PIANO_HIGH    = 108      # MIDI note 108 = C8 (highest piano key)
N_PITCHES     = PIANO_HIGH - PIANO_LOW + 1  # = 88 piano keys
MIN_ACTIVE    = 0.02     # Discard windows with < 2% active cells (sparse filter)

# ── Model Hyperparameters ─────────────────────────────────
LATENT_DIM    = 64       # Size of the compressed latent vector z
HIDDEN_DIM    = 256      # LSTM hidden state size
NUM_LAYERS    = 2        # Number of stacked LSTM layers
DROPOUT_RATE  = 0.3      # Dropout probability (prevents overfitting)

# ── Training Hyperparameters ──────────────────────────────
BATCH_SIZE    = 64       # Samples per gradient update
LEARNING_RATE = 1e-3     # Adam optimizer learning rate
NUM_EPOCHS    = 100      # Total training epochs
GRAD_CLIP     = 1.0      # Max gradient norm (prevents exploding gradients)

# ── Focal Loss Parameters ─────────────────────────────────
FOCAL_GAMMA   = 2.0      # Modulating exponent (2.0 is standard)
FOCAL_POS_W   = 20.0     # Positive class weight (compensates for ~97% sparsity)
# Note: pos_weight range 10-30 is safe per the implementation guide

# ── Generation Parameters ─────────────────────────────────
GEN_THRESHOLD = 0.3      # Binarization threshold (< 0.5 compensates for sparsity)
NUM_GENERATE  = 5        # Number of MIDI files to generate (Task 1 requires 5)

# ── Save checkpoint every N epochs ───────────────────────
SAVE_EVERY    = 5        # Save checkpoint every 5 epochs

print("\n Configuration Summary:")
print(f"  Piano-roll: {N_PITCHES} pitches × {WINDOW_LEN} time steps")
print(f"  Time resolution: {FS} fps → 1 window = {WINDOW_LEN/FS:.1f} seconds")
print(f"  Model: {NUM_LAYERS}-layer LSTM | hidden={HIDDEN_DIM} | latent={LATENT_DIM}")
print(f"  Training: {NUM_EPOCHS} epochs | batch={BATCH_SIZE} | lr={LEARNING_RATE}")
print(f"  Focal Loss: gamma={FOCAL_GAMMA} | pos_weight={FOCAL_POS_W}")
print(f"  Outputs: {OUTPUT_ROOT}")

 MAESTRO_ROOT = /content/maestro_extracted/maestro-v3.0.0
 CSV_PATH     = /content/maestro_extracted/maestro-v3.0.0/maestro-v3.0.0.csv
 Output folders ready

 Configuration Summary:
  Piano-roll: 88 pitches × 128 time steps
  Time resolution: 16 fps → 1 window = 8.0 seconds
  Model: 2-layer LSTM | hidden=256 | latent=64
  Training: 100 epochs | batch=64 | lr=0.001
  Focal Loss: gamma=2.0 | pos_weight=20.0
  Outputs: /content/drive/MyDrive/Task1_Outputs


In [ ]:
# CELL 7: Load MAESTRO CSV Metadata
df = pd.read_csv(CSV_PATH)

print(" MAESTRO Dataset Overview")
print("=" * 50)
print(f"Total recordings: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSplit distribution:")
print(df['split'].value_counts())
print(f"\nDuration statistics (seconds):")
print(df['duration'].describe().round(1))
print(f"\nComposers (top 10):")
print(df['canonical_composer'].value_counts().head(10))

# Separate splits (use official splits - do NOT re-split randomly)
df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'validation'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)

print(f"\n Done Train: {len(df_train)} recordings")
print(f" Done Validation: {len(df_val)} recordings")
print(f" Done Test: {len(df_test)} recordings")

 MAESTRO Dataset Overview
Total recordings: 1276

Columns: ['canonical_composer', 'canonical_title', 'split', 'year', 'midi_filename', 'audio_filename', 'duration']

Split distribution:
split
train         962
test          177
validation    137
Name: count, dtype: int64

Duration statistics (seconds):
count    1276.0
mean      560.5
std       443.1
min        45.2
25%       262.0
50%       429.2
75%       685.0
max      2624.7
Name: duration, dtype: float64

Composers (top 10):
canonical_composer
Frédéric Chopin            201
Franz Schubert             186
Ludwig van Beethoven       146
Johann Sebastian Bach      145
Franz Liszt                131
Sergei Rachmaninoff         59
Robert Schumann             49
Claude Debussy              45
Joseph Haydn                40
Wolfgang Amadeus Mozart     38
Name: count, dtype: int64

 Done Train: 962 recordings
 Done Validation: 137 recordings
 Done Test: 177 recordings


In [ ]:

# CELL 8: EDA Visualizations
fig = plt.figure(figsize=(16, 12))
fig.suptitle('MAESTRO Dataset — Exploratory Data Analysis',
             fontsize=16, fontweight='bold', y=1.01)

# ── Plot 1: Duration Histogram ─────────────────────────────
ax1 = fig.add_subplot(2, 3, 1)
ax1.hist(df['duration'] / 60, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax1.set_xlabel('Duration (minutes)')
ax1.set_ylabel('Number of Recordings')
ax1.set_title('Piece Duration Distribution')
ax1.axvline(df['duration'].median() / 60, color='red', linestyle='--',
            label=f'Median: {df["duration"].median()/60:.1f} min')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── Plot 2: Train/Val/Test Split ──────────────────────────
ax2 = fig.add_subplot(2, 3, 2)
split_counts = df['split'].value_counts()
colors = ['#2ecc71', '#3498db', '#e74c3c']
ax2.pie(split_counts.values, labels=split_counts.index, colors=colors,
        autopct='%1.1f%%', startangle=90)
ax2.set_title('Dataset Splits')

# ── Plot 3: Composer Distribution ─────────────────────────
ax3 = fig.add_subplot(2, 3, 3)
top_composers = df['canonical_composer'].value_counts().head(8)
ax3.barh(range(len(top_composers)), top_composers.values, color='#9b59b6', alpha=0.85)
ax3.set_yticks(range(len(top_composers)))
ax3.set_yticklabels(top_composers.index, fontsize=8)
ax3.set_xlabel('Number of Pieces')
ax3.set_title('Top 8 Composers')
ax3.grid(True, alpha=0.3, axis='x')

# ── Plot 4: Duration by Split ─────────────────────────────
ax4 = fig.add_subplot(2, 3, 4)
for split, color in zip(['train', 'validation', 'test'], colors):
    subset = df[df['split'] == split]['duration'] / 60
    ax4.hist(subset, bins=30, alpha=0.6, label=split, color=color)
ax4.set_xlabel('Duration (minutes)')
ax4.set_ylabel('Count')
ax4.set_title('Duration Distribution by Split')
ax4.legend()
ax4.grid(True, alpha=0.3)

# ── Plot 5: Year Distribution ─────────────────────────────
ax5 = fig.add_subplot(2, 3, 5)
year_counts = df['year'].value_counts().sort_index()
ax5.bar(year_counts.index.astype(str), year_counts.values, color='#e67e22', alpha=0.85)
ax5.set_xlabel('Competition Year')
ax5.set_ylabel('Number of Recordings')
ax5.set_title('Recordings per Competition Year')
ax5.tick_params(axis='x', rotation=45)
ax5.grid(True, alpha=0.3, axis='y')

# ── Plot 6: Estimated windows per recording ────────────────
ax6 = fig.add_subplot(2, 3, 6)
# Estimate: duration × fs / window_len
estimated_windows = (df['duration'] * FS / WINDOW_LEN).astype(int)
ax6.hist(estimated_windows, bins=40, color='#1abc9c', edgecolor='white', alpha=0.85)
ax6.set_xlabel('Estimated Training Windows')
ax6.set_ylabel('Count')
ax6.set_title(f'Windows per Recording\n(fs={FS}, window={WINDOW_LEN})')
ax6.grid(True, alpha=0.3)

plt.tight_layout()
eda_path = os.path.join(PLOTS_DIR, 'eda_overview.png')
plt.savefig(eda_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" EDA plot saved: {eda_path}")

 EDA plot saved: /content/drive/MyDrive/Task1_Outputs/plots/eda_overview.png


In [ ]:
# CELL 9: Pitch Distribution Analysis

def analyze_pitch_distribution(midi_files_subset, n_files=50):
    """
    Analyze pitch frequency across a sample of MIDI files.
    Returns pitch counts for all 88 piano keys.
    """
    pitch_counts = np.zeros(128)
    velocity_vals = []
    note_counts = []
    sparsity_vals = []
    errors = 0

    for midi_path in tqdm(midi_files_subset[:n_files], desc="Analyzing MIDI files"):
        try:
            pm = pretty_midi.PrettyMIDI(midi_path)
            for instrument in pm.instruments:
                for note in instrument.notes:
                    if PIANO_LOW <= note.pitch <= PIANO_HIGH:
                        pitch_counts[note.pitch] += 1
                        velocity_vals.append(note.velocity)

            # Count notes
            all_notes = [n for inst in pm.instruments for n in inst.notes]
            note_counts.append(len(all_notes))

            # Sparsity on piano-roll
            pr = pm.get_piano_roll(fs=FS)[PIANO_LOW:PIANO_HIGH+1, :]
            pr_binary = (pr > 0).astype(float)
            if pr_binary.size > 0:
                sparsity_vals.append(1.0 - pr_binary.mean())
        except Exception as e:
            errors += 1

    print(f"  Processed: {n_files - errors}/{n_files} files ({errors} errors skipped)")
    return pitch_counts, velocity_vals, note_counts, sparsity_vals

# Collect MIDI paths from training set
train_midi_paths = [
    os.path.join(MAESTRO_ROOT, row['midi_filename'])
    for _, row in df_train.iterrows()
    if os.path.exists(os.path.join(MAESTRO_ROOT, row['midi_filename']))
]
print(f"Found {len(train_midi_paths)} valid training MIDI paths")

# Run analysis on 50-file sample
pitch_counts, velocity_vals, note_counts, sparsity_vals = \
    analyze_pitch_distribution(train_midi_paths, n_files=50)

# ── Pitch Distribution Plot ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MAESTRO Piano-Roll Statistics (50-file sample)', fontsize=14, fontweight='bold')

# Pitch histogram (88 keys)
piano_pitch_counts = pitch_counts[PIANO_LOW:PIANO_HIGH+1]
pitch_labels = np.arange(PIANO_LOW, PIANO_HIGH+1)
axes[0, 0].bar(pitch_labels, piano_pitch_counts, color='steelblue', alpha=0.8, width=1.0)
axes[0, 0].set_xlabel('MIDI Pitch Number')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Pitch Distribution (88 Piano Keys)')
# Add note labels at octave boundaries
for note, label in [(21,'A0'),(33,'A1'),(45,'A2'),(57,'A3'),(69,'A4'),(81,'A5'),(93,'A6'),(105,'A7')]:
    axes[0, 0].axvline(note, color='red', alpha=0.3, linestyle='--', linewidth=0.8)
    axes[0, 0].text(note, axes[0,0].get_ylim()[1]*0.9, label, fontsize=7, rotation=90)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Velocity distribution
axes[0, 1].hist(velocity_vals, bins=50, color='#e74c3c', alpha=0.8, edgecolor='white')
axes[0, 1].set_xlabel('Velocity (0-127)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Note Velocity Distribution')
axes[0, 1].axvline(np.median(velocity_vals), color='black', linestyle='--',
                   label=f'Median: {np.median(velocity_vals):.0f}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Note count per piece
axes[1, 0].hist(note_counts, bins=30, color='#2ecc71', alpha=0.8, edgecolor='white')
axes[1, 0].set_xlabel('Notes per Recording')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Note Count Distribution')
axes[1, 0].axvline(np.median(note_counts), color='black', linestyle='--',
                   label=f'Median: {np.median(note_counts):.0f}')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Sparsity distribution
axes[1, 1].hist(sparsity_vals, bins=30, color='#9b59b6', alpha=0.8, edgecolor='white')
axes[1, 1].set_xlabel('Sparsity (fraction of zero cells)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Piano-Roll Sparsity')
axes[1, 1].axvline(np.median(sparsity_vals), color='black', linestyle='--',
                   label=f'Median: {np.median(sparsity_vals):.3f}')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
pitch_plot_path = os.path.join(PLOTS_DIR, 'pitch_distribution.png')
plt.savefig(pitch_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Pitch distribution plot saved: {pitch_plot_path}")

print(f"\n Key Statistics:")
print(f"  Average sparsity: {np.mean(sparsity_vals)*100:.1f}% zeros")
print(f"  Median note count per piece: {np.median(note_counts):.0f}")
print(f"  Median velocity: {np.median(velocity_vals):.0f}")
print(f"\n  Sparsity ~{np.mean(sparsity_vals)*100:.0f}% — this is WHY we need Focal Loss")

Found 962 valid training MIDI paths


Analyzing MIDI files: 100%|██████████| 50/50 [00:20<00:00,  2.48it/s]


  Processed: 50/50 files (0 errors skipped)
 Pitch distribution plot saved: /content/drive/MyDrive/Task1_Outputs/plots/pitch_distribution.png

 Key Statistics:
  Average sparsity: 92.2% zeros
  Median note count per piece: 6098
  Median velocity: 65

  Sparsity ~92% — this is WHY we need Focal Loss


In [ ]:
# CELL 10: Piano-Roll Preprocessing Function

def midi_to_windows(midi_path, fs=FS, window_len=WINDOW_LEN,
                    piano_low=PIANO_LOW, piano_high=PIANO_HIGH,
                    min_active=MIN_ACTIVE):
    """
    Convert a MIDI file into a list of binary piano-roll windows.

    Pipeline (4 stages from Implementation Guide):
    1. Extract piano-roll at fs frames/second → shape (128, T)
    2. Slice piano range rows 21-108 → shape (88, T)
    3. Transpose to (T, 88) — time is first axis for LSTM
    4. Binarize: set all non-zero to 1
    5. Segment into non-overlapping windows of length window_len
    6. Filter: discard windows with < min_active fraction of 1s

    Returns: list of np.arrays, each shape (window_len, 88)
    """
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)

        # Stage 1: Extract piano-roll (shape: 128 × T)
        piano_roll = pm.get_piano_roll(fs=fs)  # shape: (128, T)

        # Stage 2: Slice piano range (rows 21 to 108)
        piano_roll = piano_roll[piano_low:piano_high+1, :]  # shape: (88, T)

        # Stage 3: Transpose to (T, 88)
        piano_roll = piano_roll.T  # shape: (T, 88)

        # Stage 4: Binarize
        piano_roll = (piano_roll > 0).astype(np.float32)

        # Stage 5: Segment into non-overlapping windows
        T = piano_roll.shape[0]
        n_windows = T // window_len
        windows = []

        for i in range(n_windows):
            window = piano_roll[i * window_len : (i + 1) * window_len]  # (128, 88)

            # Stage 6: Filter sparse windows
            active_fraction = window.mean()
            if active_fraction >= min_active:
                windows.append(window)

        return windows

    except Exception as e:
        return []  # Skip malformed MIDI files silently

In [ ]:
# CELL 11: Full Preprocessing Pipeline with Disk Caching

def build_dataset_split(df_split, split_name, maestro_root=MAESTRO_ROOT,
                         processed_dir=PROCESSED_DIR):
    """
    Process all MIDI files for a given split and save as .npy file.
    Failsafe: checks if file already exists and skips reprocessing.
    """
    save_path = os.path.join(processed_dir, f'{split_name}_windows.npy')

    # ── Failsafe: Load if already processed ──────────────
    if os.path.exists(save_path):
        print(f" {split_name} data already exists at {save_path}")
        print(f"   Loading from cache...")
        data = np.load(save_path)
        print(f"   Shape: {data.shape}")
        return data

    # ── Process files ─────────────────────────────────────
    print(f"\n Processing {split_name} split ({len(df_split)} files)...")
    all_windows = []
    skipped = 0

    for _, row in tqdm(df_split.iterrows(), total=len(df_split),
                        desc=f"Processing {split_name}"):
        midi_path = os.path.join(maestro_root, row['midi_filename'])

        if not os.path.exists(midi_path):
            skipped += 1
            continue

        windows = midi_to_windows(midi_path)
        all_windows.extend(windows)

        # Intermediate save every 100 files (safety checkpoint)
        if len(all_windows) > 0 and len(all_windows) % 5000 == 0:
            temp_path = save_path.replace('.npy', '_partial.npy')
            np.save(temp_path, np.array(all_windows, dtype=np.float32))

    if len(all_windows) == 0:
        raise ValueError(f"No windows extracted for {split_name}! Check MIDI paths.")

    # Stack all windows
    data = np.array(all_windows, dtype=np.float32)

    # Save to Drive
    np.save(save_path, data)
    print(f" {split_name} complete: {data.shape[0]} windows saved to {save_path}")
    print(f"   Skipped files: {skipped}")
    print(f"   Data shape: {data.shape} "
          f"[{data.shape[0]} windows × {data.shape[1]} timesteps × {data.shape[2]} pitches]")

    return data

# ── Process all three splits ──────────────────────────────
# This saves to Google Drive
print("Starting preprocessing pipeline...")
print("=" * 60)
print("  This may take 20-40 minutes. Progress is saved to Drive.")
print("=" * 60)

train_data = build_dataset_split(df_train, 'train')
val_data   = build_dataset_split(df_val,   'validation')
test_data  = build_dataset_split(df_test,  'test')

# ── Summary ───────────────────────────────────────────────
total_windows = train_data.shape[0] + val_data.shape[0] + test_data.shape[0]
print("\n" + "=" * 60)
print(" PREPROCESSING COMPLETE")
print(f"  Train windows:      {train_data.shape[0]:>8,}  shape: {train_data.shape}")
print(f"  Validation windows: {val_data.shape[0]:>8,}  shape: {val_data.shape}")
print(f"  Test windows:       {test_data.shape[0]:>8,}  shape: {test_data.shape}")
print(f"  Total windows:      {total_windows:>8,}")
print(f"  Active fraction:    {train_data.mean()*100:.2f}% "
      f"(after sparse filtering)")

Starting preprocessing pipeline...
  This may take 20-40 minutes. Progress is saved to Drive.
 train data already exists at /content/drive/MyDrive/Task1_Outputs/processed_data/train_windows.npy
   Loading from cache...
   Shape: (62689, 128, 88)
 validation data already exists at /content/drive/MyDrive/Task1_Outputs/processed_data/validation_windows.npy
   Loading from cache...
   Shape: (7876, 128, 88)
 test data already exists at /content/drive/MyDrive/Task1_Outputs/processed_data/test_windows.npy
   Loading from cache...
   Shape: (7792, 128, 88)

 PREPROCESSING COMPLETE
  Train windows:        62,689  shape: (62689, 128, 88)
  Validation windows:    7,876  shape: (7876, 128, 88)
  Test windows:          7,792  shape: (7792, 128, 88)
  Total windows:        78,357
  Active fraction:    5.82% (after sparse filtering)


In [ ]:
# CELL 12: PyTorch Dataset Class
class PianoRollDataset(Dataset):
    """
    PyTorch Dataset for binary piano-roll windows.

    Input:  numpy array of shape (N, 128, 88)
    Output: torch tensors of shape (128, 88) per sample

    For the Autoencoder:
        - input  = the piano-roll window X
        - target = the SAME window X (we want to reconstruct it)
    """

    def __init__(self, data: np.ndarray):
        """
        Args:
            data: numpy array of shape (N, window_len, n_pitches)
        """
        # Convert to float32 tensor once (faster than converting per-sample)
        self.data = torch.tensor(data, dtype=torch.float32)
        print(f"  Dataset created: {len(self.data)} samples, "
              f"each of shape {tuple(self.data.shape[1:])}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        window = self.data[idx]  # shape: (128, 88)
        # For autoencoder: input == target (reconstruction task)
        return window, window

# ── Create Dataset objects ────────────────────────────────
print("Creating datasets...")
train_dataset = PianoRollDataset(train_data)
val_dataset   = PianoRollDataset(val_data)
test_dataset  = PianoRollDataset(test_data)

# ── Create DataLoader objects ─────────────────────────────
# num_workers=2 for parallel data loading; pin_memory speeds GPU transfer
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,           # Shuffle training data every epoch
    num_workers=2,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    drop_last=True          # Drop incomplete last batch for stable training
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,          # Never shuffle validation data
    num_workers=2,
    pin_memory=True if DEVICE.type == 'cuda' else False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True if DEVICE.type == 'cuda' else False
)

print(f"\n DataLoaders created:")
print(f"  Train:      {len(train_loader):>5} batches × {BATCH_SIZE} = "
      f"{len(train_loader) * BATCH_SIZE} samples")
print(f"  Validation: {len(val_loader):>5} batches")
print(f"  Test:       {len(test_loader):>5} batches")

# ── Verify a batch ────────────────────────────────────────
sample_x, sample_y = next(iter(train_loader))
print(f"\n Sample batch:")
print(f"  Input shape:  {tuple(sample_x.shape)} "
      f"[batch × time_steps × pitches]")
print(f"  Target shape: {tuple(sample_y.shape)}")
print(f"  Value range:  [{sample_x.min():.0f}, {sample_x.max():.0f}] "
      f"(binary: 0 or 1)")
print(f"  Active fraction in batch: {sample_x.mean()*100:.2f}%")

Creating datasets...
  Dataset created: 62689 samples, each of shape (128, 88)
  Dataset created: 7876 samples, each of shape (128, 88)
  Dataset created: 7792 samples, each of shape (128, 88)

 DataLoaders created:
  Train:        979 batches × 64 = 62656 samples
  Validation:   124 batches
  Test:         122 batches

 Sample batch:
  Input shape:  (64, 128, 88) [batch × time_steps × pitches]
  Target shape: (64, 128, 88)
  Value range:  [0, 1] (binary: 0 or 1)
  Active fraction in batch: 5.71%


In [ ]:
# CELL 13: LSTM Autoencoder Architecture

class Encoder(nn.Module):
    """
    LSTM Encoder: maps piano-roll window → latent vector z

    Input:  (batch, seq_len=128, input_size=88)
    Output: (batch, latent_dim=64)

    Key: We use h_n[-1] — the FINAL hidden state of the LAST LSTM layer.
    This is a fixed-size summary of the entire sequence.
    """

    def __init__(self, input_dim=N_PITCHES, hidden_dim=HIDDEN_DIM,
                 latent_dim=LATENT_DIM, num_layers=NUM_LAYERS,
                 dropout=DROPOUT_RATE):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Stacked LSTM layers
        # batch_first=True: input shape is (batch, seq, features) — more intuitive
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,  # Dropout between layers
            bidirectional=False  # Unidirectional for causal structure
        )

        # Dropout after LSTM output
        self.dropout = nn.Dropout(dropout)

        # Project final hidden state to latent dimension
        # hidden_dim (256) → latent_dim (64)
        self.fc_latent = nn.Linear(hidden_dim, latent_dim)

        # Layer normalization for stable training
        self.layer_norm = nn.LayerNorm(latent_dim)

    def forward(self, x):
        """
        Args:
            x: (batch, 128, 88) — batch of piano-roll windows
        Returns:
            z: (batch, 64) — latent vector for each sample
        """
        # Pass through LSTM
        # output: (batch, 128, 256) — hidden state at every time step
        # h_n:   (num_layers, batch, 256) — final hidden states
        # c_n:   (num_layers, batch, 256) — final cell states
        output, (h_n, c_n) = self.lstm(x)

        # CRITICAL: Use h_n[-1] — last layer's final hidden state
        # This is a FIXED-SIZE vector summarizing the entire sequence
        # h_n[-1] shape: (batch, 256)
        final_hidden = self.dropout(h_n[-1])

        # Project to latent space: (batch, 256) → (batch, 64)
        z = self.fc_latent(final_hidden)
        z = self.layer_norm(z)

        return z


class Decoder(nn.Module):
    """
    LSTM Decoder: maps latent vector z → reconstructed piano-roll

    Input:  (batch, latent_dim=64)
    Output: (batch, seq_len=128, input_size=88) — raw logits (no sigmoid)

    Key: z is REPEATED across all 128 time steps so the decoder
    has access to the latent code at every position throughout decoding.
    """

    def __init__(self, latent_dim=LATENT_DIM, hidden_dim=HIDDEN_DIM,
                 output_dim=N_PITCHES, seq_len=WINDOW_LEN,
                 num_layers=NUM_LAYERS, dropout=DROPOUT_RATE):
        super().__init__()

        self.seq_len = seq_len
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Project latent vector back to hidden dimension for initialization
        # AND for input at each time step
        self.latent_to_hidden = nn.Linear(latent_dim, hidden_dim)
        self.latent_to_cell   = nn.Linear(latent_dim, hidden_dim)

        # Project latent to input size so we can concatenate with z each step
        self.latent_to_input = nn.Linear(latent_dim, latent_dim)

        # LSTM receives: latent_dim (z repeated)
        self.lstm = nn.LSTM(
            input_size=latent_dim,   # z repeated across time
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

        self.dropout = nn.Dropout(dropout)

        # Output projection: hidden_dim (256) → n_pitches (88)
        #  NO sigmoid here! Raw logits → sigmoid is inside Focal Loss
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, z):
        """
        Args:
            z: (batch, 64) — latent vector
        Returns:
            logits: (batch, 128, 88) — raw scores (apply sigmoid for probabilities)
        """
        batch_size = z.size(0)

        # Initialize LSTM hidden and cell states from z
        # Expand to (num_layers, batch, hidden_dim)
        h0 = self.latent_to_hidden(z).unsqueeze(0).repeat(self.num_layers, 1, 1)
        c0 = self.latent_to_cell(z).unsqueeze(0).repeat(self.num_layers, 1, 1)

        # CRITICAL: Repeat z across all seq_len time steps
        # This injects the latent code at EVERY decoding step
        # z: (batch, 64) → decoder_input: (batch, 128, 64)
        decoder_input = self.latent_to_input(z).unsqueeze(1).repeat(1, self.seq_len, 1)

        # Decode: (batch, 128, 64) → (batch, 128, 256)
        output, _ = self.lstm(decoder_input, (h0, c0))
        output = self.dropout(output)

        # Project to pitch space: (batch, 128, 256) → (batch, 128, 88)
        logits = self.fc_out(output)  # Raw logits — NO sigmoid applied here

        return logits


class LSTMAutoencoder(nn.Module):
    """
    Full LSTM Autoencoder combining Encoder and Decoder.

    Architecture:
        Encoder: Piano-roll (128×88) → Latent z (64,)
        Decoder: Latent z (64,) → Reconstructed piano-roll (128×88)
    """

    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        """
        Args:
            x: (batch, 128, 88) input piano-roll
        Returns:
            logits: (batch, 128, 88) reconstruction logits
            z: (batch, 64) latent representation
        """
        z = self.encoder(x)
        logits = self.decoder(z)
        return logits, z

    def encode(self, x):
        """Encode only — returns latent z"""
        return self.encoder(x)

    def decode(self, z):
        """Decode only — returns logits (apply sigmoid for probabilities)"""
        return self.decoder(z)

    def generate(self, n_samples=1, device=DEVICE):
        """Generate new music by sampling z ~ N(0, I)"""
        self.eval()
        with torch.no_grad():
            z = torch.randn(n_samples, LATENT_DIM, device=device)
            logits = self.decode(z)
            probs = torch.sigmoid(logits)
        return probs.cpu().numpy()


# ── Instantiate Model ─────────────────────────────────────
model = LSTMAutoencoder().to(DEVICE)

# ── Model Summary ─────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print("  LSTM AUTOENCODER ARCHITECTURE")
print("=" * 60)
print(f"\nENCODER:")
print(f"  Input:  (batch, {WINDOW_LEN}, {N_PITCHES}) — piano-roll windows")
print(f"  LSTM:   {NUM_LAYERS} layers × {HIDDEN_DIM} hidden units")
print(f"  Output: (batch, {LATENT_DIM}) — latent vector z")
print(f"\nDECODER:")
print(f"  Input:  (batch, {LATENT_DIM}) — latent vector z")
print(f"  LSTM:   {NUM_LAYERS} layers × {HIDDEN_DIM} hidden units")
print(f"  Output: (batch, {WINDOW_LEN}, {N_PITCHES}) — reconstruction logits")
print(f"\n Parameters:")
print(f"  Total parameters:     {total_params:>12,}")
print(f"  Trainable parameters: {trainable_params:>12,}")
print(f"\n Model created on {DEVICE}")

# ── Quick forward pass test ───────────────────────────────
model.eval()
with torch.no_grad():
    test_input = torch.zeros(2, WINDOW_LEN, N_PITCHES).to(DEVICE)
    test_logits, test_z = model(test_input)
print(f"\n Forward pass test:")
print(f"  Input shape:  {tuple(test_input.shape)}")
print(f"  Latent shape: {tuple(test_z.shape)}")
print(f"  Output shape: {tuple(test_logits.shape)}")
print(f"\n Architecture verified successfully")

  LSTM AUTOENCODER ARCHITECTURE

ENCODER:
  Input:  (batch, 128, 88) — piano-roll windows
  LSTM:   2 layers × 256 hidden units
  Output: (batch, 64) — latent vector z

DECODER:
  Input:  (batch, 64) — latent vector z
  LSTM:   2 layers × 256 hidden units
  Output: (batch, 128, 88) — reconstruction logits

 Parameters:
  Total parameters:        1,813,336
  Trainable parameters:    1,813,336

 Model created on cuda

 Forward pass test:
  Input shape:  (2, 128, 88)
  Latent shape: (2, 64)
  Output shape: (2, 128, 88)

 Architecture verified successfully


In [ ]:
# CELL 14: Focal Loss Implementation

class FocalLoss(nn.Module):
    """
    Focal Loss for binary piano-roll reconstruction.

    Combines:
    - BCEWithLogitsLoss (numerically stable binary cross-entropy)
    - Focal modulating factor (1 - p_t)^gamma
    - Positive class weight (amplifies note gradient)

    Reference: "Focal Loss for Dense Object Detection" (Lin et al., 2017)

    Args:
        gamma (float): Modulating exponent. 0 = standard BCE, 2 = standard focal.
                       Higher γ = more focus on hard examples.
        pos_weight (float): Weight for positive class (active notes).
                           Should be in range 10-30 for piano-roll.
        reduction (str): 'mean' or 'sum'
    """

    def __init__(self, gamma=FOCAL_GAMMA, pos_weight=FOCAL_POS_W,
                 reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

        # Register pos_weight as buffer (moves to correct device automatically)
        pos_w = torch.tensor([pos_weight])
        self.register_buffer('pos_weight', pos_w)

    def forward(self, logits, targets):
        """
        Args:
            logits:  (batch, seq_len, n_pitches) — raw model output (NO sigmoid)
            targets: (batch, seq_len, n_pitches) — binary targets (0 or 1)
        Returns:
            scalar loss value
        """
        # Step 1: Standard BCE (numerically stable, handles sigmoid internally)
        # pos_weight must be a tensor on the same device as logits
        bce_loss = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            pos_weight=self.pos_weight,
            reduction='none'  # Keep full tensor for focal modulation
        )

        # Step 2: Compute probability p_t for focal modulation
        # p_t = probability assigned to the CORRECT class
        probs = torch.sigmoid(logits)
        # For positive cells (target=1): p_t = probs
        # For negative cells (target=0): p_t = 1 - probs
        p_t = probs * targets + (1 - probs) * (1 - targets)

        # Step 3: Focal modulating factor (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma

        # Step 4: Apply focal weight to BCE loss
        focal_loss = focal_weight * bce_loss

        # Step 5: Reduce
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


# ── Instantiate loss function ─────────────────────────────
criterion = FocalLoss(gamma=FOCAL_GAMMA, pos_weight=FOCAL_POS_W)
criterion = criterion.to(DEVICE)

# ── Verify loss function ──────────────────────────────────
print(" Testing Focal Loss:")
with torch.no_grad():
    # Test 1: All-zero prediction on all-zero target (should give low loss)
    test_logits = torch.zeros(4, WINDOW_LEN, N_PITCHES).to(DEVICE)
    test_targets = torch.zeros(4, WINDOW_LEN, N_PITCHES).to(DEVICE)
    loss_zeros = criterion(test_logits, test_targets)
    print(f"  All-zero logits, all-zero targets: {loss_zeros.item():.4f}")

    # Test 2: All-zero prediction on sparse target (should give higher loss)
    test_targets_sparse = (torch.rand(4, WINDOW_LEN, N_PITCHES) < 0.03).float().to(DEVICE)
    loss_sparse = criterion(test_logits, test_targets_sparse)
    print(f"  All-zero logits, sparse targets:   {loss_sparse.item():.4f}")

    # Test 3: Good prediction (should give very low loss)
    test_logits_good = test_targets_sparse * 5.0 - (1 - test_targets_sparse) * 5.0
    loss_good = criterion(test_logits_good, test_targets_sparse)
    print(f"  Perfect logits, sparse targets:    {loss_good.item():.4f}")

print(f"\n Focal Loss working correctly")
print(f"   Loss is higher when model misses notes (as expected)")
print(f"   gamma={FOCAL_GAMMA}, pos_weight={FOCAL_POS_W}")

 Testing Focal Loss:
  All-zero logits, all-zero targets: 0.1733
  All-zero logits, sparse targets:   0.2752
  Perfect logits, sparse targets:    0.0000

 Focal Loss working correctly
   Loss is higher when model misses notes (as expected)
   gamma=2.0, pos_weight=20.0


In [ ]:
# CELL 15: Checkpoint Save Functions

def save_checkpoint(model, optimizer, scheduler, epoch, train_losses,
                    val_losses, best_val_loss, filename='checkpoint.pt'):
    """
    Save complete training state to Google Drive.
    Saves: model weights, optimizer state, LR scheduler, epoch, loss history.
    """
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        # Save config so we can verify compatibility
        'config': {
            'latent_dim': LATENT_DIM,
            'hidden_dim': HIDDEN_DIM,
            'num_layers': NUM_LAYERS,
            'window_len': WINDOW_LEN,
            'n_pitches': N_PITCHES,
        }
    }
    path = os.path.join(CHECKPOINT_DIR, filename)
    torch.save(checkpoint, path)
    return path


def load_checkpoint(model, optimizer, scheduler, filename='checkpoint.pt'):
    """
    Load training state from Google Drive checkpoint.
    Returns: start_epoch, train_losses, val_losses, best_val_loss
    """
    path = os.path.join(CHECKPOINT_DIR, filename)

    if not os.path.exists(path):
        print(f"  No checkpoint found at {path}")
        print(f"  Starting fresh training from epoch 1")
        return 0, [], [], float('inf')

    print(f"  Found checkpoint: {path}")
    checkpoint = torch.load(path, map_location=DEVICE)

    # Restore model weights
    model.load_state_dict(checkpoint['model_state_dict'])

    # Restore optimizer state (learning rate, momentum, etc.)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Restore LR scheduler
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    start_epoch    = checkpoint['epoch'] + 1
    train_losses   = checkpoint['train_losses']
    val_losses     = checkpoint['val_losses']
    best_val_loss  = checkpoint['best_val_loss']

    print(f" Resumed from epoch {checkpoint['epoch']}")
    print(f"  Best val loss so far: {best_val_loss:.6f}")

    return start_epoch, train_losses, val_losses, best_val_loss


def train_one_epoch(model, loader, criterion, optimizer):
    """
    Run one full pass through the training set.
    Returns: average loss for this epoch
    """
    model.train()
    total_loss = 0.0
    n_batches = 0

    for batch_x, batch_y in loader:
        batch_x = batch_x.to(DEVICE, non_blocking=True)
        batch_y = batch_y.to(DEVICE, non_blocking=True)

        # Forward pass
        optimizer.zero_grad()
        logits, z = model(batch_x)

        # Compute Focal Loss
        loss = criterion(logits, batch_y)

        # Backward pass
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)

        # Parameter update
        optimizer.step()

        total_loss += loss.item()
        n_batches += 1

    return total_loss / n_batches


def validate_one_epoch(model, loader, criterion):
    """
    Run one full pass through the validation set.
    NO gradient computation — inference only.
    Returns: average validation loss
    """
    model.eval()
    total_loss = 0.0
    n_batches = 0

    with torch.no_grad():
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(DEVICE, non_blocking=True)
            batch_y = batch_y.to(DEVICE, non_blocking=True)

            logits, z = model(batch_x)
            loss = criterion(logits, batch_y)

            total_loss += loss.item()
            n_batches += 1

    return total_loss / n_batches


print(" Training functions defined")
print(f"  Checkpoint directory: {CHECKPOINT_DIR}")

 Training functions defined
  Checkpoint directory: /content/drive/MyDrive/Task1_Outputs/checkpoints


In [ ]:
# CELL 16: Optimizer, Scheduler Setup & Resume Detection

# ── Optimizer ─────────────────────────────────────────────
# Adam: adaptive learning rates per parameter, best for LSTMs
# lr=1e-3 is the recommended starting point (implementation guide)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# ── Learning Rate Scheduler ───────────────────────────────
# Reduces learning rate when validation loss stops improving
# This helps fine-tune the model in later epochs
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',        # Reduce when val_loss is not improving
    factor=0.5,        # Halve the learning rate
    patience=10,       # Wait 10 epochs before reducing
    min_lr=1e-6       # Never go below this LR
)

# ── Try to resume from checkpoint ─────────────────────────
print(" Checking for existing checkpoint...")
start_epoch, train_losses, val_losses, best_val_loss = load_checkpoint(
    model, optimizer, scheduler, filename='latest_checkpoint.pt'
)

print(f"\n Training Setup:")
print(f"  Start epoch:    {start_epoch + 1}")
print(f"  Total epochs:   {NUM_EPOCHS}")
print(f"  Remaining:      {NUM_EPOCHS - start_epoch} epochs")
print(f"  Optimizer:      Adam (lr={LEARNING_RATE})")
print(f"  LR Scheduler:   ReduceLROnPlateau (patience=10)")
print(f"  Gradient clip:  {GRAD_CLIP}")
print(f"  Save every:     {SAVE_EVERY} epochs")

 Checking for existing checkpoint...
  No checkpoint found at /content/drive/MyDrive/Task1_Outputs/checkpoints/latest_checkpoint.pt
  Starting fresh training from epoch 1

 Training Setup:
  Start epoch:    1
  Total epochs:   100
  Remaining:      100 epochs
  Optimizer:      Adam (lr=0.001)
  LR Scheduler:   ReduceLROnPlateau (patience=10)
  Gradient clip:  1.0
  Save every:     5 epochs


In [ ]:
# CELL 17: MAIN TRAINING LOOP

print(" STARTING TRAINING")
print("=" * 70)
print(f"  Device:     {DEVICE}")
print(f"  Epochs:     {start_epoch} → {NUM_EPOCHS}")
print(f"  Train size: {len(train_dataset):,} windows")
print(f"  Val size:   {len(val_dataset):,} windows")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Checkpoint: every {SAVE_EVERY} epochs → {CHECKPOINT_DIR}")
print("=" * 70)

if start_epoch >= NUM_EPOCHS:
    print(f"\n Training already complete ({NUM_EPOCHS} epochs done)")
    print("   Skip to the loss visualization cell.")
else:
    training_start_time = time.time()

    for epoch in range(start_epoch, NUM_EPOCHS):
        epoch_start = time.time()

        # ── Training pass ──────────────────────────────────
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)

        # ── Validation pass ────────────────────────────────
        val_loss = validate_one_epoch(model, val_loader, criterion)

        # ── Update LR scheduler ────────────────────────────
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # ── Record losses ──────────────────────────────────
        train_losses.append(train_loss)
        val_losses.append(val_loss)

        # ── Track best model ───────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, epoch,
                          train_losses, val_losses, best_val_loss,
                          filename='best_model.pt')

        # ── Save regular checkpoint ────────────────────────
        if (epoch + 1) % SAVE_EVERY == 0 or epoch == NUM_EPOCHS - 1:
            save_checkpoint(model, optimizer, scheduler, epoch,
                          train_losses, val_losses, best_val_loss,
                          filename='latest_checkpoint.pt')

        # ── Progress Report ────────────────────────────────
        epoch_time = time.time() - epoch_start
        elapsed_total = time.time() - training_start_time
        epochs_done = epoch - start_epoch + 1
        epochs_left = NUM_EPOCHS - epoch - 1
        eta = (elapsed_total / epochs_done) * epochs_left if epochs_done > 0 else 0

        # Print every epoch (or reduce to every 5 if too verbose)
        is_best = val_loss == best_val_loss
        print(
            f"Epoch {epoch+1:>3}/{NUM_EPOCHS} | "
            f"Train: {train_loss:.4f} | "
            f"Val: {val_loss:.4f} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {epoch_time:.1f}s | "
            f"ETA: {eta/60:.1f}min"
            + (" ★ BEST" if is_best else "")
        )

    total_time = time.time() - training_start_time
    print("\n" + "=" * 70)
    print(f" TRAINING COMPLETE")
    print(f"   Total time:     {total_time/60:.1f} minutes")
    print(f"   Best val loss:  {best_val_loss:.6f}")
    print(f"   Best model:     {os.path.join(CHECKPOINT_DIR, 'best_model.pt')}")
    print("=" * 70)

 STARTING TRAINING
  Device:     cuda
  Epochs:     0 → 100
  Train size: 62,689 windows
  Val size:   7,876 windows
  Batch size: 64
  Checkpoint: every 5 epochs → /content/drive/MyDrive/Task1_Outputs/checkpoints
Epoch   1/100 | Train: 0.2654 | Val: 0.2293 | LR: 1.00e-03 | Time: 46.6s | ETA: 76.9min ★ BEST
Epoch   2/100 | Train: 0.2254 | Val: 0.2141 | LR: 1.00e-03 | Time: 49.7s | ETA: 78.6min ★ BEST
Epoch   3/100 | Train: 0.2152 | Val: 0.2111 | LR: 1.00e-03 | Time: 51.4s | ETA: 79.6min ★ BEST
Epoch   4/100 | Train: 0.2092 | Val: 0.1997 | LR: 1.00e-03 | Time: 49.8s | ETA: 79.0min ★ BEST
Epoch   5/100 | Train: 0.1994 | Val: 0.1885 | LR: 1.00e-03 | Time: 50.5s | ETA: 78.5min ★ BEST
Epoch   6/100 | Train: 0.1899 | Val: 0.1832 | LR: 1.00e-03 | Time: 50.0s | ETA: 77.8min ★ BEST
Epoch   7/100 | Train: 0.1845 | Val: 0.1749 | LR: 1.00e-03 | Time: 50.9s | ETA: 77.3min ★ BEST
Epoch   8/100 | Train: 0.1794 | Val: 0.1741 | LR: 1.00e-03 | Time: 50.4s | ETA: 76.5min ★ BEST
Epoch   9/100 | Train: 0.1

In [ ]:

# CELL 18: Load Best Model & Plot Loss Curves

# ── Load the best saved model ─────────────────────────────
best_ckpt_path = os.path.join(CHECKPOINT_DIR, 'best_model.pt')
if os.path.exists(best_ckpt_path):
    checkpoint = torch.load(best_ckpt_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    train_losses = checkpoint['train_losses']
    val_losses   = checkpoint['val_losses']
    best_epoch   = np.argmin(val_losses)
    print(f" Best model loaded from epoch {best_epoch + 1}")
    print(f"   Best validation loss: {min(val_losses):.6f}")
    print(f"   Final train loss:     {train_losses[-1]:.6f}")
else:
    print("  Best model not found. Using current model weights.")

model = model.to(DEVICE)
model.eval()

# ── Professional Loss Curve Plot ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Task 1: LSTM Autoencoder — Training Progress',
             fontsize=15, fontweight='bold')

epochs_range = range(1, len(train_losses) + 1)

# ── Left: Full Training Curve ─────────────────────────────
ax1 = axes[0]
ax1.plot(epochs_range, train_losses, 'b-', linewidth=1.5,
         label='Training Loss', alpha=0.9)
ax1.plot(epochs_range, val_losses, 'r--', linewidth=1.5,
         label='Validation Loss', alpha=0.9)
ax1.axvline(x=best_epoch + 1, color='green', linestyle=':', linewidth=2,
            label=f'Best Model (Epoch {best_epoch+1})')
ax1.scatter([best_epoch + 1], [val_losses[best_epoch]],
            color='green', s=100, zorder=5)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Focal Loss', fontsize=12)
ax1.set_title('Training & Validation Loss')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(1, len(train_losses))

# ── Right: Log-scale view (shows early convergence better) ─
ax2 = axes[1]
ax2.semilogy(epochs_range, train_losses, 'b-', linewidth=1.5,
             label='Training Loss', alpha=0.9)
ax2.semilogy(epochs_range, val_losses, 'r--', linewidth=1.5,
             label='Validation Loss', alpha=0.9)
ax2.axvline(x=best_epoch + 1, color='green', linestyle=':', linewidth=2,
            label=f'Best Model (Epoch {best_epoch+1})')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Focal Loss (log scale)', fontsize=12)
ax2.set_title('Training & Validation Loss (Log Scale)')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, which='both')
ax2.set_xlim(1, len(train_losses))

plt.tight_layout()
loss_plot_path = os.path.join(PLOTS_DIR, 'loss_curve.png')
plt.savefig(loss_plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Loss curve saved: {loss_plot_path}")

# ── Convergence Analysis ──────────────────────────────────
print(f"\n Training Analysis:")
print(f"  Starting train loss:     {train_losses[0]:.4f}")
print(f"  Final train loss:        {train_losses[-1]:.4f}")
print(f"  Starting val loss:       {val_losses[0]:.4f}")
print(f"  Best val loss:           {min(val_losses):.4f} (epoch {best_epoch+1})")
print(f"  Train improvement:       {(1 - train_losses[-1]/train_losses[0])*100:.1f}%")

# ── Interpretation ────────────────────────────────────────
gap = abs(train_losses[-1] - val_losses[-1])
if gap < 0.05 * train_losses[-1]:
    print("\n GOOD: Train and val losses are close → No overfitting")
elif val_losses[-1] > val_losses[best_epoch] * 1.1:
    print("\n  WARNING: Validation loss rose after early epochs → Mild overfitting")
    print("   Fix: Add more dropout or use fewer LSTM layers")

 Best model loaded from epoch 98
   Best validation loss: 0.113082
   Final train loss:     0.124646
 Loss curve saved: /content/drive/MyDrive/Task1_Outputs/plots/loss_curve.png

 Training Analysis:
  Starting train loss:     0.2654
  Final train loss:        0.1246
  Starting val loss:       0.2293
  Best val loss:           0.1131 (epoch 98)
  Train improvement:       53.0%


In [ ]:
# CELL 19: Piano-Roll to MIDI Conversion

def piano_roll_to_midi(piano_roll_binary, fs=FS, tempo=120,
                        piano_low=PIANO_LOW, program=0, velocity=80):
    """
    Convert a binary piano-roll matrix to a pretty_midi.PrettyMIDI object.

    Args:
        piano_roll_binary: np.array of shape (T, 88), values 0 or 1
        fs: frames per second (same as during preprocessing)
        tempo: BPM for the output MIDI file
        piano_low: lowest MIDI pitch (21 = A0)
        program: MIDI instrument program (0 = Acoustic Grand Piano)
        velocity: note velocity (1-127, 80 is mezzo-forte)

    Returns:
        pretty_midi.PrettyMIDI object
    """
    pm = pretty_midi.PrettyMIDI(initial_tempo=tempo)
    instrument = pretty_midi.Instrument(program=program, name="Piano")

    frame_duration = 1.0 / fs  # Duration of each time step in seconds

    n_frames, n_pitches = piano_roll_binary.shape

    for pitch_idx in range(n_pitches):
        pitch = pitch_idx + piano_low  # Convert index to MIDI pitch number

        # Find where this pitch turns ON and OFF
        active = piano_roll_binary[:, pitch_idx]  # 1D array: active[t] = 1/0

        note_on = None
        for t in range(n_frames):
            if active[t] == 1 and note_on is None:
                # Note starts at time step t
                note_on = t
            elif active[t] == 0 and note_on is not None:
                # Note ends at time step t
                note_off = t
                start_time = note_on * frame_duration
                end_time   = note_off * frame_duration

                # Ensure minimum duration (avoid zero-duration notes)
                if end_time - start_time >= frame_duration:
                    note = pretty_midi.Note(
                        velocity=velocity,
                        pitch=pitch,
                        start=start_time,
                        end=end_time
                    )
                    instrument.notes.append(note)
                note_on = None

        # Handle notes active at the very end
        if note_on is not None:
            start_time = note_on * frame_duration
            end_time   = n_frames * frame_duration
            if end_time - start_time >= frame_duration:
                note = pretty_midi.Note(
                    velocity=velocity,
                    pitch=pitch,
                    start=start_time,
                    end=end_time
                )
                instrument.notes.append(note)

    # Sort notes by start time (required by pretty_midi)
    instrument.notes.sort(key=lambda x: x.start)
    pm.instruments.append(instrument)

    return pm


def verify_midi(pm, min_notes=50, min_duration=5.0):
    """
    Verify generated MIDI meets quality requirements.
    Returns: (is_valid, reason)
    """
    if pm is None:
        return False, "PrettyMIDI object is None"

    all_notes = [n for inst in pm.instruments for n in inst.notes]

    if len(all_notes) < min_notes:
        return False, f"Too few notes: {len(all_notes)} < {min_notes}"

    if pm.get_end_time() < min_duration:
        return False, f"Too short: {pm.get_end_time():.1f}s < {min_duration}s"

    return True, f"OK ({len(all_notes)} notes, {pm.get_end_time():.1f}s)"


print(" MIDI conversion functions defined")

 MIDI conversion functions defined


In [ ]:
# CELL 20: Generate 5 MIDI Samples (Task 1 Deliverable)
print(" Generating music samples...")
print("=" * 60)

model.eval()
generated_files = []
attempt = 0
max_attempts = 30  # Try up to 30 times to get 5 valid samples

with torch.no_grad():
    while len(generated_files) < NUM_GENERATE and attempt < max_attempts:
        attempt += 1

        # Step 1: Sample latent vectors z ~ N(0, I)
        # Each z encodes a different "musical idea"
        z = torch.randn(1, LATENT_DIM, device=DEVICE)

        # Step 2: Decode z → logits (batch=1, 128, 88)
        logits = model.decode(z)

        # Step 3: Sigmoid → probabilities
        probs = torch.sigmoid(logits)
        probs_np = probs.squeeze(0).cpu().numpy()  # shape: (128, 88)

        # Step 4: Binarize at threshold 0.3 (NOT 0.5 — compensates for sparsity)
        binary = (probs_np > GEN_THRESHOLD).astype(np.float32)

        # Step 5: Check if sample has enough notes
        active_frac = binary.mean()
        if active_frac < 0.005:  # < 0.5% active cells → too sparse, skip
            continue
        if active_frac > 0.5:    # > 50% active cells → too dense, skip
            continue

        # Step 6: Convert to MIDI
        pm = piano_roll_to_midi(binary, fs=FS, tempo=120, velocity=80)

        # Step 7: Verify MIDI quality
        is_valid, reason = verify_midi(pm, min_notes=50, min_duration=5.0)

        if is_valid:
            # Save to Google Drive
            filename = f'task1_generated_{len(generated_files)+1:02d}.midi'
            save_path = os.path.join(MIDI_OUT_DIR, filename)
            pm.write(save_path)
            generated_files.append(save_path)

            n_notes = len([n for inst in pm.instruments for n in inst.notes])
            print(f"   Sample {len(generated_files)}: {filename}")
            print(f"     Notes: {n_notes} | Duration: {pm.get_end_time():.1f}s "
                  f"| Active: {active_frac*100:.1f}%")
        else:
            print(f"    Attempt {attempt}: Skipped — {reason}")

print(f"\n{'=' * 60}")
print(f" Generated {len(generated_files)}/{NUM_GENERATE} valid MIDI files")
print(f"   Saved to: {MIDI_OUT_DIR}")

if len(generated_files) < NUM_GENERATE:
    print(f"\n  Generated fewer than {NUM_GENERATE} samples.")
    print(f"   Try lowering GEN_THRESHOLD to {GEN_THRESHOLD - 0.05:.2f} and re-running.")
    print(f"   Or increase NUM_EPOCHS and retrain.")

 Generating music samples...
   Sample 1: task1_generated_01.midi
     Notes: 76 | Duration: 8.0s | Active: 37.6%
   Sample 2: task1_generated_02.midi
     Notes: 85 | Duration: 8.0s | Active: 29.0%
   Sample 3: task1_generated_03.midi
     Notes: 112 | Duration: 8.0s | Active: 38.4%
   Sample 4: task1_generated_04.midi
     Notes: 50 | Duration: 8.0s | Active: 27.8%
   Sample 5: task1_generated_05.midi
     Notes: 74 | Duration: 8.0s | Active: 25.4%

 Generated 5/5 valid MIDI files
   Saved to: /content/drive/MyDrive/Task1_Outputs/generated_midi


In [ ]:
# CELL 21: Visualize Generated Piano-Rolls

def visualize_piano_roll(binary_matrix, title="Generated Piano-Roll",
                          fs=FS, piano_low=PIANO_LOW):
    """Plot a binary piano-roll matrix as a heatmap."""
    fig, ax = plt.subplots(figsize=(14, 5))

    # Transpose: show time on x-axis, pitch on y-axis
    ax.imshow(binary_matrix.T, aspect='auto', origin='lower',
              cmap='Blues', interpolation='nearest',
              extent=[0, binary_matrix.shape[0]/fs, piano_low, piano_low + binary_matrix.shape[1]])

    ax.set_xlabel('Time (seconds)', fontsize=11)
    ax.set_ylabel('MIDI Pitch', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')

    # Add piano key labels
    for note, label in [(21,'A0'),(36,'C2'),(48,'C3'),(60,'C4\n(Middle)'),(72,'C5'),(84,'C6'),(96,'C7')]:
        ax.axhline(note, color='gray', alpha=0.3, linestyle='--', linewidth=0.5)
        ax.text(-0.2/fs, note, label, fontsize=7, va='center', ha='right')

    plt.tight_layout()
    return fig

# ── Visualize 5 generated + 1 real sample ─────────────────
print("📊 Visualizing generated piano-rolls...")

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Task 1: Generated Piano-Roll Windows\n'
             '(Blue = note active, White = silence)',
             fontsize=14, fontweight='bold')

# Show 5 generated samples
model.eval()
all_generated_rolls = []

with torch.no_grad():
    for idx in range(6):
        ax = axes[idx // 2][idx % 2]

        if idx < 5:
            # Generated sample
            z = torch.randn(1, LATENT_DIM, device=DEVICE)
            logits = model.decode(z)
            probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
            binary = (probs > GEN_THRESHOLD).astype(np.float32)
            all_generated_rolls.append(binary)

            n_notes = binary.sum()
            active = binary.mean() * 100
            ax.imshow(binary.T, aspect='auto', origin='lower', cmap='Blues',
                      interpolation='nearest')
            ax.set_title(f'Generated Sample {idx+1}  '
                        f'[{n_notes:.0f} active cells, {active:.1f}% density]',
                        fontsize=10)
        else:
            # Real sample for comparison
            real_sample = train_data[np.random.randint(len(train_data))]
            n_notes = real_sample.sum()
            active = real_sample.mean() * 100
            ax.imshow(real_sample.T, aspect='auto', origin='lower', cmap='Oranges',
                      interpolation='nearest')
            ax.set_title(f'Real MAESTRO Sample (Reference)  '
                        f'[{n_notes:.0f} active cells, {active:.1f}% density]',
                        fontsize=10)

        ax.set_xlabel('Time Steps')
        ax.set_ylabel('Pitch Index (0=A0, 87=C8)')

plt.tight_layout()
pianoroll_path = os.path.join(PLOTS_DIR, 'generated_piano_rolls.png')
plt.savefig(pianoroll_path, dpi=150, bbox_inches='tight')
plt.show()
print(f" Piano-roll visualization saved: {pianoroll_path}")

📊 Visualizing generated piano-rolls...
 Piano-roll visualization saved: /content/drive/MyDrive/Task1_Outputs/plots/generated_piano_rolls.png


In [ ]:
# CELL 22: Evaluation Metric Functions

def compute_pitch_histogram_similarity(midi_path_gen, midi_path_ref=None,
                                        ref_pitch_dist=None):
    """
    Compute L1 distance between pitch class histograms.

    Maps each note to its pitch class: pitch % 12 (C, C#, D, ..., B)
    Normalizes both histograms, computes L1 distance.

    Score 0 = identical pitch class usage
    Score 2 = maximum possible difference
    Lower is better (more similar to real music)
    """
    def get_pitch_distribution(midi_path):
        try:
            pm = pretty_midi.PrettyMIDI(midi_path)
            counts = np.zeros(12)
            for inst in pm.instruments:
                for note in inst.notes:
                    counts[note.pitch % 12] += 1
            total = counts.sum()
            if total > 0:
                counts = counts / total
            return counts
        except:
            return np.ones(12) / 12  # Uniform if can't load

    gen_dist = get_pitch_distribution(midi_path_gen)

    if ref_pitch_dist is not None:
        ref_dist = ref_pitch_dist
    elif midi_path_ref is not None:
        ref_dist = get_pitch_distribution(midi_path_ref)
    else:
        raise ValueError("Must provide either ref_pitch_dist or midi_path_ref")

    # L1 distance
    similarity = np.sum(np.abs(gen_dist - ref_dist))
    return similarity, gen_dist


def compute_rhythm_diversity(midi_path, quantize_ms=50):
    """
    Rhythm Diversity Score = #unique_durations / #total_notes

    Note durations are quantized to 50ms bins to avoid floating-point noise.
    Higher score = more rhythmic variety (better).
    """
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)
        durations = []
        for inst in pm.instruments:
            for note in inst.notes:
                dur_seconds = note.end - note.start
                # Quantize to nearest 50ms
                dur_quantized = round(dur_seconds * 1000 / quantize_ms) * quantize_ms
                durations.append(dur_quantized)

        if len(durations) == 0:
            return 0.0

        unique = len(set(durations))
        total  = len(durations)
        return unique / total
    except:
        return 0.0


def compute_repetition_ratio(midi_path, ngram_n=4):
    """
    Repetition Ratio = #repeated_n-grams / #total_n-grams

    Extracts pitch sequence sorted by onset time.
    Computes all overlapping 4-grams.
    Counts n-grams that appear more than once.

    Range [0, 1]:
    - 0 = completely unique (incoherent noise)
    - 1 = completely repetitive (degenerate output)
    - 0.1-0.5 = typical for coherent classical music (ideal)
    """
    try:
        pm = pretty_midi.PrettyMIDI(midi_path)

        # Get all notes sorted by onset time
        all_notes = []
        for inst in pm.instruments:
            all_notes.extend(inst.notes)
        all_notes.sort(key=lambda n: n.start)

        if len(all_notes) < ngram_n:
            return 0.0

        # Extract pitch sequence
        pitches = [n.pitch for n in all_notes]

        # Compute overlapping n-grams
        ngrams = [tuple(pitches[i:i+ngram_n]) for i in range(len(pitches)-ngram_n+1)]

        if len(ngrams) == 0:
            return 0.0

        # Count n-grams appearing more than once
        counts = Counter(ngrams)
        repeated = sum(1 for c in counts.values() if c > 1)
        total = len(ngrams)

        return repeated / total
    except:
        return 0.0


def compute_reconstruction_loss(model, loader, criterion, n_batches=20):
    """
    Compute average Focal Loss on a data loader (validation or test set).
    Uses only n_batches for speed.
    """
    model.eval()
    losses = []
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= n_batches:
                break
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits, _ = model(x)
            loss = criterion(logits, y)
            losses.append(loss.item())
    return np.mean(losses)


print(" Evaluation metric functions defined")
print("\nMetrics:")
print("  • Pitch Histogram Similarity (L1 distance, lower = better)")
print("  • Rhythm Diversity Score (unique/total durations, higher = better)")
print("  • Repetition Ratio (repeated n-grams, ideal: 0.1–0.5)")
print("  • Reconstruction Loss (Focal Loss on validation set)")

 Evaluation metric functions defined

Metrics:
  • Pitch Histogram Similarity (L1 distance, lower = better)
  • Rhythm Diversity Score (unique/total durations, higher = better)
  • Repetition Ratio (repeated n-grams, ideal: 0.1–0.5)
  • Reconstruction Loss (Focal Loss on validation set)


In [ ]:
# CELL 23: Evaluate Task 1 Model

# ── Step 1: Compute Reconstruction Loss ───────────────────
print(" Computing evaluation metrics...")
print("-" * 50)

recon_loss_val  = compute_reconstruction_loss(model, val_loader,  criterion, n_batches=50)
recon_loss_test = compute_reconstruction_loss(model, test_loader, criterion, n_batches=50)
print(f"Reconstruction Loss (val):  {recon_loss_val:.4f}")
print(f"Reconstruction Loss (test): {recon_loss_test:.4f}")

# ── Step 2: Compute Reference Pitch Distribution ──────────
# Use a few real MIDI files as reference
print("\nBuilding reference pitch distribution from real MAESTRO files...")
ref_pitch_dists = []
for _, row in list(df_test.iterrows())[:10]: # Fix: Convert to list before slicing
    midi_path = os.path.join(MAESTRO_ROOT, row['midi_filename'])
    if os.path.exists(midi_path):
        try:
            pm = pretty_midi.PrettyMIDI(midi_path)
            counts = np.zeros(12)
            for inst in pm.instruments:
                for note in inst.notes:
                    counts[note.pitch % 12] += 1
            total = counts.sum()
            if total > 0:
                ref_pitch_dists.append(counts / total)
        except:
            pass

if ref_pitch_dists:
    ref_pitch_dist = np.mean(ref_pitch_dists, axis=0)
    print(f"  Built from {len(ref_pitch_dists)} reference files")
else:
    ref_pitch_dist = np.ones(12) / 12
    print("  Warning: Using uniform reference distribution")

# ── Step 3: Evaluate Generated MIDI Files ────────────────
print(f"\nEvaluating {len(generated_files)} generated MIDI files...")
pitch_sims = []
rhythm_divs = []
rep_ratios = []

for midi_path in generated_files:
    if not os.path.exists(midi_path):
        continue

    sim, _ = compute_pitch_histogram_similarity(midi_path, ref_pitch_dist=ref_pitch_dist)
    rdiv   = compute_rhythm_diversity(midi_path)
    rratio = compute_repetition_ratio(midi_path)

    pitch_sims.append(sim)
    rhythm_divs.append(rdiv)
    rep_ratios.append(rratio)

    print(f"  {os.path.basename(midi_path)}: "
          f"PitchSim={sim:.3f}  RhythmDiv={rdiv:.3f}  RepRatio={rratio:.3f}")

print(f"\n TASK 1 MODEL — AVERAGE METRICS:")
print(f"  Reconstruction Loss:       {recon_loss_val:.4f}")
print(f"  Pitch Histogram Similarity:{np.mean(pitch_sims):.4f}  (lower = more similar to real)")
print(f"  Rhythm Diversity:          {np.mean(rhythm_divs):.4f}  (higher = more varied)")
print(f"  Repetition Ratio:          {np.mean(rep_ratios):.4f}  (ideal: 0.1–0.5)")

# Store results for comparison table
task1_metrics = {
    'model': 'Task 1: LSTM Autoencoder',
    'recon_loss': recon_loss_val,
    'pitch_sim': np.mean(pitch_sims),
    'rhythm_div': np.mean(rhythm_divs),
    'rep_ratio': np.mean(rep_ratios),
}

 Computing evaluation metrics...
--------------------------------------------------
Reconstruction Loss (val):  0.1215
Reconstruction Loss (test): 0.1180

Building reference pitch distribution from real MAESTRO files...
  Built from 10 reference files

Evaluating 5 generated MIDI files...
  task1_generated_01.midi: PitchSim=0.192  RhythmDiv=0.553  RepRatio=0.000
  task1_generated_02.midi: PitchSim=0.339  RhythmDiv=0.576  RepRatio=0.000
  task1_generated_03.midi: PitchSim=0.224  RhythmDiv=0.473  RepRatio=0.000
  task1_generated_04.midi: PitchSim=0.454  RhythmDiv=0.460  RepRatio=0.000
  task1_generated_05.midi: PitchSim=0.342  RhythmDiv=0.541  RepRatio=0.000

 TASK 1 MODEL — AVERAGE METRICS:
  Reconstruction Loss:       0.1215
  Pitch Histogram Similarity:0.3105  (lower = more similar to real)
  Rhythm Diversity:          0.5206  (higher = more varied)
  Repetition Ratio:          0.0000  (ideal: 0.1–0.5)
